In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
env = dbutils.widgets.get("env")

In [0]:
# API_KEY = "REMOVED_SECRET"
# BASE_URL = "https://api.data.gov.in/resource/3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69"

# # Get today's date in DD-MM-YYYY format
# today_date = datetime.now().strftime("%d-%m-%Y")

# limit = 100
# offset = 0

# all_records = []

# while True:
#     params = {
#         "api-key": API_KEY,
#         "format": "json",
#         "limit": limit,
#         "offset": offset,
#         "country": "India"
#     }

#     response = requests.get(BASE_URL, params=params)
    
#     # Check if response is successful and contains content
#     if response.status_code != 200:
#         print(f"API returned status code {response.status_code} at offset {offset}. Stopping.")
#         break
    
#     if not response.text:
#         print(f"Empty response at offset {offset}. Stopping.")
#         break
    
#     try:
#         data = response.json()
#     except requests.exceptions.JSONDecodeError:
#         print(f"Invalid JSON response at offset {offset}. Response text: {response.text[:200]}")
#         break
    
#     records = data.get("records", [])

#     if not records:
#         break

#     all_records.extend(records)
#     print(f"Fetched {len(records)} | Total: {len(all_records)}")

#     offset += limit

API returned status code 502 at offset 0. Stopping.


In [0]:
import requests
import pandas as pd
import time
from datetime import datetime

API_KEY = "REMOVED_SECRET"

BASE_URL = "https://api.data.gov.in/resource/3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69"

limit = 100
offset = 0

all_records = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

while True:

    params = {
        "api-key": API_KEY,
        "format": "json",
        "limit": limit,
        "offset": offset
    }

    retry = 3
    data_finished = False
    for attempt in range(retry):

        try:

            response = requests.get(
                BASE_URL,
                params=params,
                headers=headers,
                timeout=30
            )

            print(f"Status Code: {response.status_code}")
            if response.status_code == 200:
                data = response.json()
                records = data.get("records", [])

                # STOP CONDITION
                if not records:
                    print("No more records found")
                    data_finished = True
                    break

                all_records.extend(records)
                print(f"Fetched {len(records)} | Total: {len(all_records)}")
                offset += limit
                time.sleep(1)
                break

            else:
                print(f"Retry {attempt+1} -> Status: {response.status_code}")
                time.sleep(5)

        except Exception as e:
            print(f"Error: {e}")
            time.sleep(5)

    # OUTER LOOP BREAK
    if data_finished:
        break

print(f"\nTotal Records Collected: {len(all_records)}")

Status Code: 200
Fetched 100 | Total: 100
Status Code: 200
Fetched 100 | Total: 200
Status Code: 200
Fetched 100 | Total: 300
Status Code: 200
Fetched 100 | Total: 400
Status Code: 200
Fetched 100 | Total: 500
Status Code: 200
Fetched 100 | Total: 600
Status Code: 200
Fetched 100 | Total: 700
Status Code: 200
Fetched 100 | Total: 800
Status Code: 200
Fetched 100 | Total: 900
Status Code: 200
Fetched 100 | Total: 1000
Status Code: 200
Fetched 100 | Total: 1100
Status Code: 200
Fetched 100 | Total: 1200
Status Code: 200
Fetched 100 | Total: 1300
Status Code: 200
Fetched 100 | Total: 1400
Status Code: 200
Fetched 100 | Total: 1500
Status Code: 200
Fetched 100 | Total: 1600
Status Code: 200
Fetched 100 | Total: 1700
Status Code: 200
Fetched 100 | Total: 1800
Status Code: 200
Fetched 100 | Total: 1900
Status Code: 200
Fetched 100 | Total: 2000
Status Code: 200
Fetched 100 | Total: 2100
Status Code: 200
Fetched 100 | Total: 2200
Status Code: 200
Fetched 100 | Total: 2300
Status Code: 200
Fet

In [0]:
# %sql
# create schema dev_catalog.aqi_strm_dev

In [0]:
# %sql
    
# create external table dev_catalog.aqi_strm_dev.aqi (
#   avg_value STRING,
#   city STRING,
#   country STRING,
#   last_update STRING,
#   latitude STRING,
#   longitude STRING,
#   max_value STRING,
#   min_value STRING,
#   pollutant_id STRING,
#   state STRING,
#   station STRING
#   )
# USING delta
# TBLPROPERTIES (
#   'delta.enableDeletionVectors' = 'true',
#   'delta.feature.appendOnly' = 'supported',
#   'delta.feature.deletionVectors' = 'supported',
#   'delta.feature.invariants' = 'supported',
#   'delta.minReaderVersion' = '3',
#   'delta.minWriterVersion' = '7',
#   'delta.parquet.compression.codec' = 'zstd')
# location "abfss://raw@dltdatalake.dfs.core.windows.net/aqi_raw"

In [0]:
df = spark.createDataFrame(all_records)
df.write.mode("append").saveAsTable(f"{env}_catalog.aqi_strm_{env}.aqi")